# 5. Memory Module (Ollama) 핵심 요약

이 노트북에서는 LangChain에서 대화 내역(Context)을 기억하고 관리하는 다양한 **Memory 모듈**과 이를 모델에 적용하는 방법을 다룹니다.

---

## 🧠 메모리 종류

1. **`ConversationBufferMemory` (5.0)**
   - **특징**: 모든 대화 기록을 원본 그대로 저장하고 반환합니다.
   - **장단점**: 가장 단순하지만 대화가 길어지면 토큰 길이가 무한히 늘어나 비용과 컨텍스트 제한을 초과할 수 있습니다.

2. **`ConversationBufferWindowMemory` (5.1)**
   - **특징**: 설정한 `k` 횟수(Human-AI 왕복 횟수)만큼의 최근 대화만 기억합니다.
   - **장단점**: 제한된 길이로 토큰 초과 문제는 방지할 수 있지만, 지정된 범위를 벗어난 과거의 대화 문맥은 잊어버립니다.

3. **`ConversationSummaryMemory` (5.2)**
   - **특징**: LLM을 사용하여 전체 대화 내역 전체를 하나의 지속적인 요약문(Summary)으로 압축하여 기억합니다.
   - **장단점**: 엄청나게 긴 대화에서도 메모리를 짧게 유지할 수 있으나, 매번 요약을 업데이트하기 위해 추가적인 LLM 호출(비용/시간)이 발생합니다.

4. **`ConversationSummaryBufferMemory` (5.3)** ✨ *(가장 널리 쓰이는 권장 방식)*
   - **특징**: Buffer와 Summary 방식을 결합하여 두 방식의 단점을 상쇄합니다.
   - **동작**: 최근 대화는 생생하게 버퍼에 보존하다가 누적 토큰이 `max_token_limit`를 초과하면 가장 오래된 메시지부터 차례대로 요약버전으로 변환하여 저장합니다.

5. **`ConversationKGMemory` (5.4)**
   - **특징**: 대화 속 문맥에서 핵심 주체(Entity)와 그들 간의 관계를 지식 그래프(Knowledge Graph) 형태로 추출하여 기억합니다.
   - **참고**: 관계 추출 및 구조화 능력이 뛰어나야 하므로 Llama 3.1(8B) 같은 소형 로컬 모델보다는 GPT-4 등 강력한 상용 모델의 사용이 권장됩니다.

---

## 🛠️ 모델 및 메모리 체인 통합 기법

1. **`LLMChain` & 텍스트 템플릿 주입 (5.5)**
   - 가장 전통적인 방식입니다. `PromptTemplate` 내에 `{chat_history}` 와 같은 텍스트 변수 자리를 만들어 메모리의 내용을 주입합니다.

2. **Chat 모델용 `MessagesPlaceholder` (5.6)**
   - `ChatPromptTemplate`과 `MessagesPlaceholder(variable_name="chat_history")`를 결합하여 구조화된 HumanMessage, AIMessage 객체 리스트를 안전하고 정확하게 프롬프트의 중간중간에 삽입합니다.

3. **`LCEL` (LangChain Expression Language) 체이닝 (5.7)**
   - 최신 LangChain 파이프라인 방식입니다.(`|` 연산자 사용)
   - `RunnablePassthrough.assign(history=load_memory)`를 이용하여 데이터가 흐르는 실행 과정 중에 동적으로 메모리를 로드해 모델로 전달하는 현대적인 패턴을 따릅니다.

In [1]:
# 5.0 ConversationBufferMemory

from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)
# return_message=False 혹은 매개변수가 없으면 그냥 string 형태로 반환
# return_message=True를 넣으면 Chat model을 위한 Message 형태로 반환됨

memory.save_context({"input": "Hi!"}, {"output": "How are you?"})
memory.save_context({"input": "Hi!"}, {"output": "How are you?"})
memory.save_context({"input": "Hi!"}, {"output": "How are you?"})
memory.load_memory_variables({})
# 대화가 길어질수록 memory에 내용이 계속 쌓임


/tmp/ipykernel_173877/3224045711.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(return_messages=True)


{'history': [HumanMessage(content='Hi!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='How are you?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Hi!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='How are you?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Hi!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='How are you?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
# 5.1 ConversationBufferWindowMemory

from langchain_classic.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=4
)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

add_message("1", "1")
add_message("2", "2")
add_message("3", "3")
add_message("4", "4")
add_message("5", "5")

memory.load_memory_variables({})

/tmp/ipykernel_173877/4076439076.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


{'history': [HumanMessage(content='2', additional_kwargs={}, response_metadata={}),
  AIMessage(content='2', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='3', additional_kwargs={}, response_metadata={}),
  AIMessage(content='3', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='4', additional_kwargs={}, response_metadata={}),
  AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='5', additional_kwargs={}, response_metadata={}),
  AIMessage(content='5', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
# 5.2 ConversationSummaryMemory

from langchain_classic.memory import ConversationSummaryMemory
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
)

memory = ConversationSummaryMemory(llm=llm)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

def get_history():
    return memory.load_memory_variables({})

add_message("Hi I'm Nicolas, I live in South Korea", "Wow that is so cool!")
add_message("South Korea is so preety", "I wish I could go!!!")

get_history()

/tmp/ipykernel_175727/1044144679.py:10: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm)


{'history': 'Here\'s the updated summary:\n\nNicolas introduces himself as living in South Korea, which sparks enthusiasm from the AI who thinks it would be great to visit. Nicolas also describes South Korea as "so pretty", with the AI expressing a desire to see its beauty firsthand.\n\nLet me know if you\'d like to add more lines of conversation!'}

In [2]:
# 5.3 ConversationSummaryBufferMemory

from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=150,
    return_messages=True
)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

def get_history():
    return memory.load_memory_variables({})

add_message("Hi I'm Nicolas, I live in South Korea", "Wow that is so cool!")
add_message("South Korea is so pretty", "I wish I could go!!!")
add_message("How far is Korea from Argentina?", "I don't know! Super far!")
add_message("How far is Korea from Argentina?", "I don't know! Super far!")
add_message("How far is Korea from Argentina?", "I don't know! Super far!")
add_message("How far is Korea from Argentina?", "I don't know! Super far!")
add_message("How far is Korea from Argentina?", "I don't know! Super far!")
add_message("How far is Korea from Argentina?", "I don't know! Super far!")
add_message("How far is Korea from Argentina?", "I don't know! Super far!")

get_history()

{'history': [SystemMessage(content="Here's the updated summary:\n\nNicolas introduces himself as being from South Korea. The AI finds this interesting and thinks it's cool.", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='South Korea is so pretty', additional_kwargs={}, response_metadata={}),
  AIMessage(content='I wish I could go!!!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='How far is Korea from Argentina?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="I don't know! Super far!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='How far is Korea from Argentina?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="I don't know! Super far!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='How far is Korea from Argentina?', additional_kwargs={}, respon

In [ ]:
# 5.4 ConversationKGMemory

from langchain_classic.memory import ConversationKGMemory
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
)

memory = ConversationKGMemory(
    llm=llm,
    return_messages=True,
)

def add_message(input, output):
    memory.save_context({"input": input}, {"output": output})

add_message("Hi I'm Nicolas, I live in South Korea", "Wow that is so cool!")
add_message("Nicolas likes kimchi", "Wow that is so cool!")

memory.load_memory_variables({"input": "who is Nicolas"})
memory.load_memory_variables({"input": "what does nicolas like"})

# llama3.1 모델에서는 지식 그래프를 생성하지 못함. GPT-4와 같은 모델 사용 필요

{'history': []}

In [15]:
# 5.5 Memory on LLMChain

from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_ollama import ChatOllama
from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate

llm = ChatOllama(
    model="llama3.1",
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history"
)

template = """
    You are a helpful AI talking to a human.

    {chat_history}
    Human: {question}
    You:
"""

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=PromptTemplate.from_template(template),
    verbose=True,
)

chain.invoke({"question": "My name is Nico"})



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    
    Human: My name is Nico
    You:


> Finished chain.


{'question': 'My name is Nico',
 'chat_history': '',
 'text': "Hello Nico! It's nice to meet you. How can I assist or chat with you today?"}

In [16]:
chain.invoke({"question": "I live in Seoul"})



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Nico
AI: Hello Nico! It's nice to meet you. How can I assist or chat with you today?
    Human: I live in Seoul
    You:


> Finished chain.


{'question': 'I live in Seoul',
 'chat_history': "Human: My name is Nico\nAI: Hello Nico! It's nice to meet you. How can I assist or chat with you today?",
 'text': "Seoul! What a vibrant city! What would you like to talk about, Nico? Are you looking for recommendations on what to do or see while living there? Or perhaps you'd like to discuss the local culture, food, or even learn some basic Korean phrases? I'm all ears (or rather, all text)!"}

In [17]:
chain.invoke({"question": "What is my name?"})



> Entering new LLMChain chain...
Prompt after formatting:

    You are a helpful AI talking to a human.

    Human: My name is Nico
AI: Hello Nico! It's nice to meet you. How can I assist or chat with you today?
Human: I live in Seoul
AI: Seoul! What a vibrant city! What would you like to talk about, Nico? Are you looking for recommendations on what to do or see while living there? Or perhaps you'd like to discuss the local culture, food, or even learn some basic Korean phrases? I'm all ears (or rather, all text)!
    Human: What is my name?
    You:


> Finished chain.


{'question': 'What is my name?',
 'chat_history': "Human: My name is Nico\nAI: Hello Nico! It's nice to meet you. How can I assist or chat with you today?\nHuman: I live in Seoul\nAI: Seoul! What a vibrant city! What would you like to talk about, Nico? Are you looking for recommendations on what to do or see while living there? Or perhaps you'd like to discuss the local culture, food, or even learn some basic Korean phrases? I'm all ears (or rather, all text)!",
 'text': "Nice to refresh your memory! Your name is Nico. We've only just started chatting, and I'm here to help with anything you'd like to discuss about Seoul or life in general!"}

In [18]:
memory.load_memory_variables({})

{'chat_history': "System: The human introduces themselves as Nico, and the AI responds by greeting them and inquiring about their intentions for chatting. \n\n(To be continued...)\nHuman: I live in Seoul\nAI: Seoul! What a vibrant city! What would you like to talk about, Nico? Are you looking for recommendations on what to do or see while living there? Or perhaps you'd like to discuss the local culture, food, or even learn some basic Korean phrases? I'm all ears (or rather, all text)!\nHuman: What is my name?\nAI: Nice to refresh your memory! Your name is Nico. We've only just started chatting, and I'm here to help with anything you'd like to discuss about Seoul or life in general!"}

In [19]:
# 5.6 Chat Based Memory

from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_ollama import ChatOllama
from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder

llm = ChatOllama(
    model="llama3.1",
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    memory_key="chat_history",
    return_messages=True
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI talking to a human"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=True,
)

chain.invoke({"question": "My name is Nico"})



> Entering new LLMChain chain...
Prompt after formatting:
System: You are a helpful AI talking to a human
Human: My name is Nico

> Finished chain.


{'question': 'My name is Nico',
 'chat_history': [HumanMessage(content='My name is Nico', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Nice to meet you, Nico! How's your day going so far? Is there anything on your mind that you'd like to chat about or perhaps get some assistance with? I'm all ears (or rather, all text)!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'text': "Nice to meet you, Nico! How's your day going so far? Is there anything on your mind that you'd like to chat about or perhaps get some assistance with? I'm all ears (or rather, all text)!"}

In [20]:
chain.invoke({"question": "I live in Seoul"})



> Entering new LLMChain chain...
Prompt after formatting:
System: You are a helpful AI talking to a human
Human: My name is Nico
AI: Nice to meet you, Nico! How's your day going so far? Is there anything on your mind that you'd like to chat about or perhaps get some assistance with? I'm all ears (or rather, all text)!
Human: I live in Seoul

> Finished chain.


{'question': 'I live in Seoul',
 'chat_history': [HumanMessage(content='My name is Nico', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Nice to meet you, Nico! How's your day going so far? Is there anything on your mind that you'd like to chat about or perhaps get some assistance with? I'm all ears (or rather, all text)!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='I live in Seoul', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Seoul is an amazing city. What part of Seoul do you call home? Are you a local born and raised, or did you move here from somewhere else?\n\nBy the way, have you tried any good food lately? Korean cuisine is one of my favorite things about the country!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'text': 'Seoul is an amazing city. What part of Seoul do you call home? Are you a local born and raised, or did you move her

In [21]:
chain.invoke({"question": "What is my name?"})



> Entering new LLMChain chain...
Prompt after formatting:
System: You are a helpful AI talking to a human
Human: My name is Nico
AI: Nice to meet you, Nico! How's your day going so far? Is there anything on your mind that you'd like to chat about or perhaps get some assistance with? I'm all ears (or rather, all text)!
Human: I live in Seoul
AI: Seoul is an amazing city. What part of Seoul do you call home? Are you a local born and raised, or did you move here from somewhere else?

By the way, have you tried any good food lately? Korean cuisine is one of my favorite things about the country!
Human: What is my name?

> Finished chain.


{'question': 'What is my name?',
 'chat_history': [HumanMessage(content='I live in Seoul', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Seoul is an amazing city. What part of Seoul do you call home? Are you a local born and raised, or did you move here from somewhere else?\n\nBy the way, have you tried any good food lately? Korean cuisine is one of my favorite things about the country!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Your name is Nico. I remember! We were just talking about Seoul and how amazing it is there. What do you like to do in your free time, Nico?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'text': 'Your name is Nico. I remember! We were just talking about Seoul and how amazing it is there. What do you like to do in your free time, Nico?'}

In [34]:
# 5.7 LCEL Based Memory

from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_ollama.chat_models import ChatOllama
from langchain_classic.schema.runnable import RunnablePassthrough
from langchain_classic.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOllama(
    model="llama3.1",
    temperature=0.1,
)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ]
)

def load_memory(input):
    return memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

def invoke_chain(question):
    result = chain.invoke({
        # "history": load_memory(),
        "question": "My name is Nico"
    })
    memory.save_context(
        {"input": question}, 
        {"output": result.content},
    )
    print(result.content)

In [35]:
invoke_chain("My name is nico")

Nice to meet you, Nico! How's your day going so far? Is there anything on your mind that you'd like to chat about or ask for help with? I'm all ears (or rather, all text)!


In [36]:
invoke_chain("What is my name?")

You mentioned that already, but I'll say it back: Nice to meet you, Nico! What's new and exciting in your world right now? Want to talk about something specific or just see where the conversation takes us?
